# 00b — Probe the three script-based datasets

`datasets` 4.x no longer runs loading scripts, so BanglaNMT, ELRC-Medical-V2 and
MultiEURLEX failed in `00_setup_check`. This notebook finds another way in:

- **A.** HuggingFace's auto-converted Parquet copy (branch `refs/convert/parquet`), loaded with the current `datasets`.
- **B.** The raw files in each repo, and the download URLs inside each loading script.
- **C.** *(only if A failed)* the old `datasets` 3.6 with `trust_remote_code=True`, installed in a side folder so the main environment is untouched.

CPU is enough (no GPU needed). **Internet: On**, secret `HF_TOKEN` attached.
At the end it writes `/kaggle/working/script_datasets_report.json` — send that file back.

In [ ]:
import json, re
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, login
from datasets import load_dataset

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)

REPOS = ["csebuetnlp/BanglaNMT", "qanastek/ELRC-Medical-V2", "coastalcph/multi_eurlex"]
# Configs we care about, if they exist under these names
WANTED = {
    "csebuetnlp/BanglaNMT": [],
    "qanastek/ELRC-Medical-V2": ["en-fr", "en-hr", "en-it", "en-pl", "en-de", "en-es", "en-pt", "en-fi"],
    "coastalcph/multi_eurlex": ["all_languages", "en"],
}
report = {}

## A. Auto-converted Parquet branch

In [ ]:
def first_row(repo, config, split):
    ds = load_dataset(repo, config, split=split, revision="refs/convert/parquet", streaming=True)
    row = next(iter(ds))
    return list(row.keys()), str(row)[:500]

for repo in REPOS:
    entry = {}
    try:
        files = [f for f in api.list_repo_files(repo, repo_type="dataset", revision="refs/convert/parquet")
                 if f.endswith(".parquet")]
        layout = {}
        for f in files:
            parts = f.split("/")
            cfg = parts[0]
            split = parts[1] if len(parts) > 2 else re.sub(r"\.parquet$", "", parts[-1]).split("-")[-1]
            layout.setdefault(cfg, {}).setdefault(split, 0)
            layout[cfg][split] += 1
        entry["configs"] = sorted(layout)
        entry["splits_per_config"] = {c: sorted(s) for c, s in layout.items()}
        try_cfgs = [c for c in WANTED[repo] if c in layout] or sorted(layout)[:1]
        entry["samples"] = {}
        for cfg in try_cfgs:
            split = "test" if "test" in layout[cfg] else sorted(layout[cfg])[0]
            try:
                cols, ex = first_row(repo, cfg, split)
                entry["samples"][f"{cfg}/{split}"] = {"columns": cols, "example": ex}
            except Exception as e:
                entry["samples"][f"{cfg}/{split}"] = {"error": f"{type(e).__name__}: {e}"[:300]}
    except Exception as e:
        entry["error"] = f"{type(e).__name__}: {e}"[:300]
    report.setdefault(repo, {})["A_parquet_branch"] = entry
    print(repo, json.dumps(entry, ensure_ascii=False, indent=1)[:2500], "\n")

## B. Raw repo files and the URLs each loading script downloads from

In [ ]:
for repo in REPOS:
    entry = {}
    try:
        files = api.list_repo_files(repo, repo_type="dataset")
        entry["files"] = files[:60]
        entry["n_files"] = len(files)
        scripts = [f for f in files if f.endswith(".py")]
        urls = []
        for s in scripts:
            text = open(hf_hub_download(repo, s, repo_type="dataset", token=HF_TOKEN)).read()
            urls += sorted(set(re.findall(r"https?://[^\s'\"\)]+", text)))
        entry["script_urls"] = urls[:40]
    except Exception as e:
        entry["error"] = f"{type(e).__name__}: {e}"[:300]
    report[repo]["B_raw"] = entry
    print(repo, json.dumps(entry, ensure_ascii=False, indent=1)[:2500], "\n")

## C. Optional — old `datasets` with the loading script

Run this only if section A printed errors for a dataset. It installs `datasets==3.6.0`
into a side folder and probes in a separate process. MultiEURLEX downloads a large archive,
so it can take 10+ minutes; it is tried last.

In [ ]:
import subprocess, sys, textwrap, os

need_c = [r for r in REPOS if not report[r]["A_parquet_branch"].get("samples")
          or all("error" in v for v in report[r]["A_parquet_branch"]["samples"].values())]
print("Need fallback C for:", need_c)

if need_c:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", "/kaggle/working/ds3",
                    "datasets==3.6.0"], check=True)
    probe = textwrap.dedent('''
        import json, sys
        from datasets import load_dataset, get_dataset_config_names
        repo = sys.argv[1]
        out = {}
        try:
            cfgs = get_dataset_config_names(repo, trust_remote_code=True)
            out["configs"] = cfgs
            wanted = {"qanastek/ELRC-Medical-V2": "en-fr", "coastalcph/multi_eurlex": "all_languages"}
            cfg = wanted.get(repo) if wanted.get(repo) in cfgs else (cfgs[0] if cfgs else None)
            ds = load_dataset(repo, cfg, trust_remote_code=True)
            out["config_used"] = cfg
            out["splits"] = {k: len(v) for k, v in ds.items()}
            first = ds[list(ds)[0]][0]
            out["columns"] = list(first.keys())
            out["example"] = str(first)[:500]
        except Exception as e:
            out["error"] = f"{type(e).__name__}: {e}"[:500]
        print("RESULT" + json.dumps(out, ensure_ascii=False))
    ''')
    open("/kaggle/working/probe_ds3.py", "w").write(probe)
    env = dict(os.environ, PYTHONPATH="/kaggle/working/ds3", HF_TOKEN=HF_TOKEN)
    for repo in sorted(need_c, key=lambda r: r == "coastalcph/multi_eurlex"):
        p = subprocess.run([sys.executable, "/kaggle/working/probe_ds3.py", repo],
                           capture_output=True, text=True, env=env, timeout=3600)
        line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT")), None)
        result = json.loads(line[6:]) if line else {"error": (p.stderr or p.stdout)[-500:]}
        report[repo]["C_datasets_3_6"] = result
        print(repo, json.dumps(result, ensure_ascii=False, indent=1)[:2500], "\n")

## Save the report

In [ ]:
with open("/kaggle/working/script_datasets_report.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(open("/kaggle/working/script_datasets_report.json").read())